# 오디오 전처리 파이프라인 v2 (yusung)

대상 원본 데이터: `data/raw/{asvspoof2019_la, commonvoice_ko, musan, fakemusiccaps}`

이 노트북은 위 4개 데이터셋을 평가 데이터 스펙(문제 설명 2번 참고)에 맞춰 **하나의 표준 포맷**으로 변환하고,
`VP/MP/VF/MF/FF` 5개 라벨과 **train/val/test split**을 부여한 뒤, 추후 음성+음악 혼합 데이터를 만들기 쉬운
구조로 `yusung/data_processed_v2/`에 저장합니다.

> **v2 변경점**: 기존 버전(`yusung/data_processed/`, 노트북 실행 이력은 그대로 보존)은 10초 미만 자투리를
> **무조건 zero-pad**했습니다. v2는 데이터셋 특성에 따라 이 규칙을 3가지로 분기합니다 (2절 `chunk_audio` 참고).
> 기존 결과물은 건드리지 않도록 출력 폴더를 `data_processed_v2/`로 분리했습니다.

## 라벨 규칙

| 데이터셋 | 구분 기준 | VP | MP | VF | MF | FF |
|---|---|---|---|---|---|---|
| ASVspoof 2019 LA | bonafide | 1 | 0 | 0 | -1 | 0 |
| ASVspoof 2019 LA | spoof | 1 | 0 | 1 | -1 | 1 |
| Common Voice Korean | 모든 음성 | 1 | 0 | 0 | -1 | 0 |
| MUSAN | speech/ | 1 | 0 | 0 | -1 | 0 |
| MUSAN | music/ | 0 | 1 | -1 | 0 | 0 |
| MUSAN | noise/ | 0 | 0 | -1 | -1 | 0 |
| FakeMusicCaps | 모든 음악 | 0 | 1 | -1 | 1 | 1 |

- 1: 해당 특성이 존재하거나 fake / 0: 해당 특성이 없거나 real / -1: 해당 성분 자체가 없어서 판정 대상이 아님
  → 학습 시 `known_mask = labels != -1` 로 loss에서 제외.
- 이 5개 라벨은 평가 예측 항목과 다음과 같이 대응합니다: `FF→FILE_FAKE_PROB`, `VF→VOICE_FAKE_PROB`,
  `MF→MUSIC_FAKE_PROB`, `VP→VOICE_PRESENT_PROB`, `MP→MUSIC_PRESENT_PROB`.
- chunk 파일은 원본 파일의 라벨을 그대로 상속합니다 (`data/metadata/02_build_mvp_metadata.ipynb`와 동일 컨벤션).

## 10초 청크 분할 규칙 (v2, 데이터셋별 분기)

| 데이터셋 | 원본 특성 | chunk 정책 | 10초 미만 자투리 처리 | 이유 |
|---|---|---|---|---|
| ASVspoof2019 LA | 평균 3.53초, 대부분 짧은 발화 | `single_chunk` (원본 1파일 → 1 sample) | zero-pad | 짧은 utterance 자체가 정상 데이터 특성. loop하면 같은 발화를 반복하는 비현실적 패턴 발생 |
| Common Voice KO | 평균 5.64초, 전부 10초 미만 | `single_chunk` (원본 1파일 → 1 sample) | zero-pad | 실제 짧은 발화 데이터. ASVspoof와 동일하게 반복 금지 |
| MUSAN speech | 58~692초 장시간 | `sequential_drop` (10초 순차 분할) | 마지막 remainder 버림 | 원본이 충분히 길어 손실 극소. 마지막 몇 초짜리 조각을 굳이 패딩할 이유 없음 |
| MUSAN music | 75~702초 장시간 | `sequential_drop` (10초 순차 분할) | 마지막 remainder 버림 | 동일. 긴 원본에서 충분한 clean 10초 chunk 확보 가능 |
| FakeMusicCaps | 생성 자체가 약 10.0~10.24초 | `single_chunk` (앞 10초만 사용) | 0.xx초 remainder 버림 | 사실상 numerical/encoding tail. 별도 sample로 만드는 것이 명백히 부적절 |
| MUSAN noise | 0.7~398초, 길이 편차 큼 | `noise_conditional` (≥10초는 sequential_drop, <10초는 single_chunk) | 길이에 따라 조건부 처리 | 편차가 커서 파일 단위로 다른 규칙 필요 |

즉 **음성/음악 어느 쪽도 loop padding은 쓰지 않는다** — 짧은 원본은 zero-pad로 실제 길이 특성을 보존하고,
충분히 긴 원본(MUSAN speech/music/noise≥10s)은 자투리를 버려서 무음이 섞인 청크를 만들지 않는다.

## 평가 데이터 스펙에 맞추기 위해 이 노트북이 하는 일

1. **포맷 표준화**: 모든 확장자(wav/flac/mp3/...)를 로드해 **16kHz / mono**로 통일.
   (평가셋은 모노·스테레오가 혼재하지만, 학습 데이터는 mono로 표준화합니다 — 이유는 맨 아래 "설계 결정과 한계" 참고.)
2. **라우드니스(dB) 정규화**: 모든 소스가 서로 다른 녹음/마스터링 레벨을 갖고 있으므로, 추후 음성+음악을 섞을 때
   한쪽이 항상 더 크거나 작지 않도록 **LUFS 기반 라우드니스**를 동일 목표치로 맞춥니다.
3. **10초 단위 청크 분할 (데이터셋별 정책)**: 위 표의 `chunk_policy`(`single_chunk`/`sequential_drop`/
   `noise_conditional`)에 따라 분기합니다. 자세한 로직은 2절 `chunk_audio` 참고.
4. **train/val/test split**: 같은 화자/클립이 여러 split에 걸쳐 새지 않도록 그룹 단위로 분리.
5. **혼합 친화적 메타데이터**: 오디오 저장 구조와 split 배정을 분리하고, VP/MP 기준으로 걸러낸
   `voice_pool.csv` / `music_pool.csv`를 별도로 만들어 추후 음성+음악 혼합 데이터 생성 시 바로 사용할 수 있게 합니다.
6. **체크포인트/재개(resume)**: 처리된 chunk는 파일 단위로 즉시 `chunks_partial.csv`에 append됩니다.
   실행 중간에 커널이 죽어도(연결 끊김, 서버 재시작 등) 이미 처리된 파일은 다시 계산하지 않고, 노트북을
   처음부터 다시 실행하면 남은 파일부터 이어서 처리합니다 (2절 `load_done_source_paths`/`append_rows_checkpoint`).

전화채널 오디오, 다양한 확장자 등 평가셋의 나머지 특성에 대한 대응은 맨 아래 "설계 결정과 한계"에서 설명합니다.


## 0. 환경 준비

이 레포의 `deepvoice` conda 커널에는 `librosa`, `soundfile`, `pandas`, `numpy`, `tqdm`, `scikit-learn`이
이미 설치되어 있습니다. 라우드니스 정규화에 필요한 `pyloudnorm`만 없으므로, 없을 때만 설치합니다.

In [ ]:
import importlib
import subprocess
import sys

for pkg in ["pyloudnorm"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json
import math
import os
import warnings
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import pyloudnorm as pyln
import soundfile as sf
from tqdm.auto import tqdm

# pyloudnorm은 게인 적용 후 peak가 1.0을 넘으면 미리 경고를 띄우는데, 우리는 바로 다음 단계
# (normalize_loudness의 peak_ceiling 클램프)에서 항상 그 초과분을 눌러주므로 안전하게 무시한다.
warnings.filterwarnings("ignore", message="Possible clipped samples in output")

## 1. 설정값 (Config)

여기서 조정하는 값들이 전체 파이프라인의 동작을 결정합니다. `MAX_FILES_PER_DATASET`은 코드가 제대로
동작하는지 빠르게 확인하고 싶을 때 작은 값(예: 20)으로 바꿔서 먼저 돌려보는 용도입니다. 실제 전체 처리를
할 때는 반드시 `None`으로 되돌리세요.

`NUM_WORKERS`는 이 서버가 다른 팀원들과 공유하는 4-core 머신이라는 점을 감안해 `cpu_count - 1`로 보수적으로
잡았습니다. 다른 사람이 무거운 작업을 돌리고 있다면 더 낮춰서 사용하세요.

`PARTIAL_CSV`는 체크포인트 파일입니다. 이 노트북은 파일 하나를 처리할 때마다 결과를 바로 이 CSV에
append하므로, 실행 도중(네트워크 끊김, 서버 재시작 등으로) 커널이 죽어도 이미 처리된 파일은 재실행 시
자동으로 건너뜁니다 — 처음부터 다시 실행해도 됩니다.

`CHUNK_POLICY_BY_KEY`는 v2에서 새로 추가된 설정으로, 데이터셋(MUSAN은 서브카테고리까지)별로 어떤 자투리
처리 정책(`single_chunk`/`sequential_drop`/`noise_conditional`)을 쓸지 정의합니다 (위 표 참고).
`OUT_ROOT`도 기존 `data_processed/`와 섞이지 않도록 `data_processed_v2/`로 분리했습니다.


In [ ]:
# ===== 경로 =====
RAW_ROOT = Path("/cluster/workspace/deepvoice/data/raw")
OUT_ROOT = Path("/cluster/workspace/deepvoice/yusung/data_processed_v2")  # v1(data_processed/)과 섞이지 않도록 분리
AUDIO_OUT = OUT_ROOT / "audio"        # 표준화된 오디오 (16k/mono/loudness-normalized/10s chunk)
META_OUT = OUT_ROOT / "metadata"      # 라벨 + split이 담긴 csv들
AUDIO_OUT.mkdir(parents=True, exist_ok=True)
META_OUT.mkdir(parents=True, exist_ok=True)

# ===== 오디오 표준화 =====
TARGET_SR = 16000                     # 평가 데이터 스펙: 16kHz
CHUNK_SEC = 10                        # 10초 단위 청크
CHUNK_SAMPLES = TARGET_SR * CHUNK_SEC

# ===== 라우드니스 정규화 (EBU R128 기반 LUFS) =====
TARGET_LUFS = -23.0                   # 방송 표준(EBU R128) 기본값. 모든 소스를 이 라우드니스로 통일.
PEAK_CEILING_DB = -1.0                # 정규화 후 true-peak이 이 값을 넘지 않도록 안전 마진(클리핑 방지)
PEAK_CEILING = 10 ** (PEAK_CEILING_DB / 20)

# ===== 10초 미만/이상 자투리 처리 정책 (v2에서 새로 추가, 위 표 참고) =====
# "single_chunk": 파일당 1개 청크만 생성, 길면 앞만 crop, 짧으면 zero-pad
# "sequential_drop": 10초 단위 순차 분할, 마지막 자투리는 버림(패딩 없음)
# "noise_conditional": 10초 이상이면 sequential_drop, 미만이면 single_chunk
CHUNK_POLICY_BY_KEY = {
    "asvspoof2019_la": "single_chunk",
    "commonvoice_ko": "single_chunk",
    "musan/speech": "sequential_drop",
    "musan/music": "sequential_drop",
    "musan/noise": "noise_conditional",
    "fakemusiccaps": "single_chunk",
}

# ===== train/val/test =====
SPLIT_RATIOS = (0.8, 0.1, 0.1)        # (train, val, test)
SPLIT_SEED = 42

# ===== 실행 =====
NUM_WORKERS = max(1, (os.cpu_count() or 4) - 1)   # 공유 서버이므로 1코어는 남겨둠
MAX_FILES_PER_DATASET = None          # 디버그용 상한. 빠른 검증 시 예: 20. 실제 실행 시 반드시 None.

AUDIO_EXTS = (".wav", ".flac", ".mp3", ".m4a", ".ogg")

# ===== 메타데이터 컬럼 순서 + 체크포인트 파일 =====
COLUMN_ORDER = [
    "chunk_id", "filepath", "dataset", "split", "orig_split", "category", "source_model",
    "VP", "MP", "VF", "MF", "FF",
    "chunk_policy",
    "chunk_index", "n_chunks_in_file", "is_padded", "pad_seconds", "is_truncated",
    "orig_duration_sec", "orig_sr", "orig_channels",
    "measured_lufs", "target_lufs", "gain_applied_db", "peak_after_norm", "loudness_norm_applied",
    "group_id", "source_path",
]

# 처리된 chunk 메타데이터를 파일 단위로 즉시 append하는 체크포인트. 커널이 도중에 죽어도
# 여기까지 기록된 내용은 디스크에 남아있어서, 재실행 시 이미 끝난 파일을 다시 처리하지 않는다 (2절 참고).
PARTIAL_CSV = META_OUT / "chunks_partial.csv"

print(f"TARGET_SR={TARGET_SR}, CHUNK_SEC={CHUNK_SEC}, TARGET_LUFS={TARGET_LUFS}, "
      f"NUM_WORKERS={NUM_WORKERS}, MAX_FILES_PER_DATASET={MAX_FILES_PER_DATASET}")
print(f"OUT_ROOT={OUT_ROOT}")
if PARTIAL_CSV.exists():
    n_done = sum(1 for _ in open(PARTIAL_CSV)) - 1
    print(f"체크포인트 발견: {PARTIAL_CSV} ({n_done}개 chunk 기록됨) -> 해당 파일들은 자동으로 건너뜁니다.")


## 2. 오디오 처리 핵심 함수

파일 하나를 "로드 → mono/16kHz → 라우드니스 정규화 → 10초 청크 분할(정책별 분기) → 저장"까지 처리하는
`_process_file_job` 하나로 모든 데이터셋을 동일하게 처리합니다. (데이터셋마다 다른 부분은 "어떤 파일을,
어떤 라벨/`chunk_policy`로, 어디에 저장할지"뿐이며 이건 3~6절에서 job 딕셔너리로 만들어 넘겨줍니다.)

`chunk_audio(y, policy)`가 v2의 핵심 변경점입니다. 정책 3종(`single_chunk`/`sequential_drop`/
`noise_conditional`)에 대한 설명은 1절 표와 함수 docstring을 참고하세요. 이제 각 청크는
`(arr, is_padded, pad_seconds, is_truncated)` 4-튜플을 반환합니다 — `is_truncated`는 앞부분만 잘라내고
뒤를 버린 경우(FakeMusicCaps, 긴 ASVspoof/CommonVoice 예외 케이스)를 표시하는 v2 신규 컬럼입니다.

`ProcessPoolExecutor`로 병렬 처리하기 때문에, worker 함수는 클로저 없이 **top-level 함수 + 순수 인자**로만
동작하도록 작성했습니다 (pickle 가능해야 하므로).

여기서 `load_done_source_paths`/`append_rows_checkpoint` 두 함수가 체크포인트/재개(resume)를 담당합니다:
데이터셋별 job을 만들기 전에 `load_done_source_paths()`로 "이미 끝난 원본 파일 목록"을 읽어와 그 파일들은
job에서 아예 제외하고, `run_jobs`가 끝나자마자 결과를 `append_rows_checkpoint`로 바로 디스크에 씁니다.
그래서 3~6절 어디서 실행이 끊기든, 노트북을 처음부터 재실행하면 이미 끝난 부분은 몇 초 만에 건너뛰고
남은 파일부터 이어서 처리합니다.


In [ ]:
def probe_audio_info(path):
    """원본 파일의 샘플레이트/채널수/길이를 (오디오를 통째로 디코딩하지 않고) 빠르게 조회."""
    try:
        info = sf.info(str(path))
        duration = info.frames / info.samplerate if info.samplerate else None
        return {"samplerate": info.samplerate, "channels": info.channels, "duration_sec": duration}
    except Exception:
        return {"samplerate": None, "channels": None, "duration_sec": None}


def load_mono_16k(path, target_sr=TARGET_SR):
    """
    오디오를 mono/16kHz로 로드.
    librosa.load(sr=..., mono=True)가 리샘플링과 (스테레오 -> 채널 평균) 모노 변환을 함께 수행.
    soundfile 백엔드가 mp3/flac/wav/ogg/m4a를 모두 지원하므로 확장자별 분기가 필요 없다.
    """
    try:
        y, sr = librosa.load(str(path), sr=target_sr, mono=True)
        return y.astype(np.float32), sr
    except Exception as e:
        return None, str(e)


def normalize_loudness(y, sr, target_lufs=TARGET_LUFS, peak_ceiling=PEAK_CEILING):
    """
    ITU-R BS.1770 (EBU R128) integrated loudness를 측정해 target_lufs로 맞춘다.
    이후 true peak이 peak_ceiling을 넘으면 선형으로 눌러서 클리핑을 방지한다
    (라우드니스가 target보다 살짝 낮아질 수 있지만, 이게 표준적인 "loudness normalize + peak limit" 방식).

    아주 짧거나(< 0.4초, BS.1770 게이팅 블록 최소 길이) 무음에 가까운 신호는 측정치가
    -inf/NaN이 되거나 예외가 나므로, 이런 경우 정규화를 건너뛰고 원신호를 그대로 반환한다
    (버리지 않고 살려서 chunk 단계로 넘긴다).
    """
    if y is None or len(y) < int(sr * 0.4):
        peak = float(np.max(np.abs(y))) if y is not None and len(y) else 0.0
        return y, None, 0.0, peak, False

    meter = pyln.Meter(sr)
    try:
        measured = meter.integrated_loudness(y)
    except Exception:
        return y, None, 0.0, float(np.max(np.abs(y))), False

    if not math.isfinite(measured):
        return y, None, 0.0, float(np.max(np.abs(y))), False

    gain_db = target_lufs - measured
    y_norm = pyln.normalize.loudness(y, measured, target_lufs)

    peak = float(np.max(np.abs(y_norm))) if len(y_norm) else 0.0
    if peak > peak_ceiling:
        y_norm = y_norm * (peak_ceiling / peak)
        peak = peak_ceiling

    return y_norm.astype(np.float32), float(measured), float(gain_db), peak, True


def chunk_audio(y, policy, chunk_samples=CHUNK_SAMPLES):
    """
    청크 분할 정책 3종 (v2에서 데이터셋별로 분기; v1은 전부 "항상 zero-pad, 절대 안 버림"이었음):

      - "single_chunk": 파일당 정확히 1개 청크만 생성. 10초보다 길면 앞 10초만 사용(뒤는 버림,
        is_truncated=True), 10초보다 짧으면 뒤를 0으로 패딩(is_padded=True). ASVspoof/CommonVoice
        (원본 자체가 짧은 실제 발화)와 FakeMusicCaps(생성 자체가 ~10.0~10.24초라 자투리가 사실상
        인코딩 tail)에 사용. 원본을 loop해서 반복 패턴을 만드는 대신, 짧으면 있는 그대로 zero-pad.
      - "sequential_drop": 10초 단위로 순차 분할하고, 마지막 10초 미만 자투리는 패딩하지 않고
        그냥 버림(청크를 만들지 않음). 원본이 충분히 길어(MUSAN speech/music, 58~702초) 자투리를
        버려도 데이터 손실이 미미할 때 사용.
      - "noise_conditional": 원본 길이가 10초 이상이면 sequential_drop과 동일하게, 10초 미만이면
        single_chunk와 동일하게 처리. MUSAN noise처럼 길이 편차가 커서(0.7~398초) 파일별로
        조건부 처리가 필요한 경우 사용.

    반환: [(arr, is_padded, pad_seconds, is_truncated), ...]
    """

    def _single(y_):
        n = len(y_)
        if n == chunk_samples:
            return [(y_, False, 0.0, False)]
        if n > chunk_samples:
            return [(y_[:chunk_samples], False, 0.0, True)]
        pad_width = chunk_samples - n
        padded = np.pad(y_, (0, pad_width), mode="constant")
        return [(padded, True, pad_width / TARGET_SR, False)]

    def _sequential_drop(y_):
        n_full = len(y_) // chunk_samples
        if n_full == 0:
            # 안전장치: 이 정책 대상 데이터셋에서는 발생하지 않아야 하지만, 혹시 예상보다 짧은
            # 파일이 섞여 있어도 전체를 버리지 않고 zero-pad 1개로 보존한다.
            return _single(y_)
        chunks = [
            (y_[i * chunk_samples:(i + 1) * chunk_samples], False, 0.0, False)
            for i in range(n_full)
        ]
        # 마지막 자투리(remainder < chunk_samples)는 의도적으로 버림: 패딩도, 별도 청크 생성도 안 함.
        return chunks

    if policy == "single_chunk":
        return _single(y)
    if policy == "sequential_drop":
        return _sequential_drop(y)
    if policy == "noise_conditional":
        return _sequential_drop(y) if len(y) >= chunk_samples else _single(y)
    raise ValueError(f"unknown chunk_policy: {policy!r}")


def _process_file_job(job):
    """
    job = {"in_path", "out_dir", "base_name", "label": {..., "chunk_policy": ...}} 하나를 처리해서
    {"rows": [chunk별 메타데이터 dict, ...], "error": str | None} 을 반환.
    ProcessPoolExecutor.map에 그대로 넘길 수 있도록 순수 함수로 작성.
    """
    in_path = job["in_path"]
    out_dir = Path(job["out_dir"])
    base_name = job["base_name"]
    label = job["label"]
    chunk_policy = label["chunk_policy"]

    try:
        info = probe_audio_info(in_path)
        y, sr = load_mono_16k(in_path)
        if y is None or len(y) == 0:
            return {"rows": [], "error": f"{in_path}: load failed ({sr})"}

        y_norm, measured_lufs, gain_db, peak, norm_ok = normalize_loudness(y, sr)
        chunks = chunk_audio(y_norm, chunk_policy)

        out_dir.mkdir(parents=True, exist_ok=True)
        rows = []
        for idx, (arr, is_padded, pad_seconds, is_truncated) in enumerate(chunks):
            out_path = out_dir / f"{base_name}_chunk{idx:04d}.wav"
            arr = np.clip(arr, -1.0, 1.0)  # PCM_16 저장 전 최종 안전장치 (부동소수점 오차로 인한 오버슈트 방지)
            sf.write(str(out_path), arr, TARGET_SR, subtype="PCM_16")

            rel_path = out_path.relative_to(AUDIO_OUT)
            # chunk_id를 dataset+base_name만으로 만들면 FakeMusicCaps처럼 동일 파일명(원곡 id)이
            # 여러 하위 폴더(생성모델)에 반복되는 경우 충돌한다. 실제로 유일함이 보장된 상대경로에서
            # 그대로 파생시켜 folder 구분이 항상 반영되도록 한다.
            chunk_id = str(rel_path.with_suffix("")).replace("/", "__")

            row = dict(label)
            row.update(
                chunk_id=chunk_id,
                filepath=str(rel_path),  # AUDIO_OUT 기준 상대경로로 저장 (이식성)
                chunk_index=idx,
                n_chunks_in_file=len(chunks),
                is_padded=is_padded,
                pad_seconds=round(pad_seconds, 4),
                is_truncated=is_truncated,
                orig_duration_sec=info["duration_sec"],
                orig_sr=info["samplerate"],
                orig_channels=info["channels"],
                measured_lufs=measured_lufs,
                target_lufs=TARGET_LUFS,
                gain_applied_db=gain_db,
                peak_after_norm=peak,
                loudness_norm_applied=norm_ok,
            )
            rows.append(row)
        return {"rows": rows, "error": None}
    except Exception as e:
        return {"rows": [], "error": f"{in_path}: {e}"}


def run_jobs(jobs, desc, num_workers=NUM_WORKERS):
    """job 리스트를 (병렬로) 처리하고 모든 chunk 행을 하나의 리스트로 모아 반환."""
    all_rows, errors = [], []
    if num_workers <= 1:
        results = (_process_file_job(j) for j in jobs)
        for res in tqdm(results, total=len(jobs), desc=desc):
            all_rows.extend(res["rows"])
            if res["error"]:
                errors.append(res["error"])
    else:
        with ProcessPoolExecutor(max_workers=num_workers) as ex:
            for res in tqdm(ex.map(_process_file_job, jobs, chunksize=8), total=len(jobs), desc=desc):
                all_rows.extend(res["rows"])
                if res["error"]:
                    errors.append(res["error"])
    if errors:
        print(f"[{desc}] {len(errors)}개 실패, 예시:", errors[:3])
    return all_rows


def load_done_source_paths():
    """체크포인트(PARTIAL_CSV)에 이미 기록된 source_path 집합을 반환. 재실행 시 이 파일들은 건너뛴다."""
    if not PARTIAL_CSV.exists():
        return set()
    return set(pd.read_csv(PARTIAL_CSV, usecols=["source_path"])["source_path"])


def append_rows_checkpoint(rows):
    """새로 처리된 chunk 행들을 PARTIAL_CSV에 즉시 append (파일이 없을 때만 헤더를 쓴다).
    이 함수가 끝나는 순간 디스크에 기록되므로, 그 이후 커널이 죽어도 이 rows는 재계산할 필요가 없다."""
    if not rows:
        return
    chunk_df = pd.DataFrame(rows)[COLUMN_ORDER]
    chunk_df.to_csv(PARTIAL_CSV, mode="a", header=not PARTIAL_CSV.exists(), index=False)


## 3. ASVspoof 2019 LA

`ASVspoof2019_LA_{train,dev,eval}/flac/*.flac`를 각 split의 protocol 파일로 bonafide/spoof를 조회합니다.
ASVspoof는 화자 단위로 이미 train/dev/eval이 겹치지 않게 설계된 공인 benchmark split이므로, 이 노트북에서
새로 무작위 분할하지 않고 **원본 split을 그대로 존중**해 `train→train, dev→val, eval→test`로 매핑합니다.
(`group_id`에 화자 id를 남겨두어, 나중에 필요하면 재분할할 수도 있게 합니다.)

In [ ]:
ASV_ROOT = RAW_ROOT / "asvspoof2019_la" / "extracted" / "LA"
ASV_PROTOCOLS = ASV_ROOT / "ASVspoof2019_LA_cm_protocols"

ASV_SPLITS = {
    "train": ("ASVspoof2019_LA_train", "ASVspoof2019.LA.cm.train.trn.txt", "train"),
    "dev":   ("ASVspoof2019_LA_dev",   "ASVspoof2019.LA.cm.dev.trl.txt",   "val"),
    "eval":  ("ASVspoof2019_LA_eval",  "ASVspoof2019.LA.cm.eval.trl.txt",  "test"),
}

done = load_done_source_paths()  # 이전(중단된) 실행에서 이미 끝낸 원본 파일들 -> 이번엔 건너뛴다

asv_rows = []
asv_unmapped = 0
asv_skipped = 0

for orig_split, (folder, proto_name, mapped_split) in ASV_SPLITS.items():
    proto_df = pd.read_csv(
        ASV_PROTOCOLS / proto_name, sep=r"\s+", header=None,
        names=["speaker_id", "file_id", "unused", "system_id", "key"],
    )
    proto_map = {r.file_id: (r.key, r.speaker_id) for r in proto_df.itertuples()}

    flac_dir = ASV_ROOT / folder / "flac"
    files = sorted(flac_dir.glob("*.flac"))[:MAX_FILES_PER_DATASET]

    jobs = []
    for f in files:
        if str(f) in done:
            asv_skipped += 1
            continue
        info = proto_map.get(f.stem)
        if info is None:
            asv_unmapped += 1
            continue
        key, speaker_id = info
        is_spoof = int(key == "spoof")
        label = dict(
            dataset="asvspoof2019_la", VP=1, MP=0, VF=is_spoof, MF=-1, FF=is_spoof,
            source_model=None, category=key,
            chunk_policy=CHUNK_POLICY_BY_KEY["asvspoof2019_la"],
            orig_split=orig_split, split=mapped_split,
            group_id=f"asvspoof2019_la::{speaker_id}", source_path=str(f),
        )
        jobs.append(dict(
            in_path=str(f), out_dir=str(AUDIO_OUT / "asvspoof2019_la" / orig_split),
            base_name=f.stem, label=label,
        ))

    new_rows = run_jobs(jobs, desc=f"asvspoof2019_la/{orig_split}")
    append_rows_checkpoint(new_rows)  # 이 split이 끝나는 대로 바로 디스크에 기록 (eval처럼 오래 걸리는 split 대비)
    asv_rows += new_rows

print("asvspoof2019_la 신규 chunk 수:", len(asv_rows), "| 이어서 건너뜀:", asv_skipped, "| protocol 매칭 실패:", asv_unmapped)


## 4. Common Voice Korean

전부 사람이 녹음한 실제 음성입니다 (`VP=1, MP=0, VF=0, MF=-1, FF=0`).

Common Voice는 화자 겹침을 막기 위해 자체적으로 `train.tsv`/`dev.tsv`/`test.tsv`를 제공하므로, 이 공식
split에 포함된 파일은 그대로 사용합니다 (`train.tsv→train, dev.tsv→val, test.tsv→test`). 공식 split에
없는 나머지 파일들(예: `other.tsv`에만 있는 클립)은 `client_id`(화자) 기준으로 그룹 분할합니다
(7절의 `grouped_split` 참고) — 같은 화자의 음성이 train과 val/test에 동시에 들어가는 걸 방지하기 위함입니다.

In [ ]:
CV_ROOT = RAW_ROOT / "commonvoice_ko" / "cv-corpus-26.0-2026-06-12" / "ko"

# client_id 조회용 lookup: 모든 tsv를 합쳐서 path -> client_id 매핑을 만든다.
tsv_files = ["validated.tsv", "train.tsv", "dev.tsv", "test.tsv", "other.tsv", "invalidated.tsv"]
lookup_frames = [
    pd.read_csv(CV_ROOT / name, sep="\t", usecols=["client_id", "path"])
    for name in tsv_files if (CV_ROOT / name).exists()
]
client_lookup = (
    pd.concat(lookup_frames)
    .drop_duplicates(subset="path", keep="first")
    .set_index("path")["client_id"]
    .to_dict()
)

# 공식 split 매핑 (train/dev/test.tsv에 있는 파일만)
official_split_map = {}
for name, mapped in [("train.tsv", "train"), ("dev.tsv", "val"), ("test.tsv", "test")]:
    p = CV_ROOT / name
    if p.exists():
        for path in pd.read_csv(p, sep="\t", usecols=["path"])["path"]:
            official_split_map[path] = mapped

done = load_done_source_paths()

cv_files = sorted((CV_ROOT / "clips").glob("*.mp3"))[:MAX_FILES_PER_DATASET]
cv_jobs = []
cv_skipped = 0
for f in cv_files:
    if str(f) in done:
        cv_skipped += 1
        continue
    client_id = client_lookup.get(f.name, f.stem)  # tsv에 없으면 자기 자신을 그룹으로
    split = official_split_map.get(f.name)  # 없으면 None -> 7절에서 grouped_split으로 채움
    label = dict(
        dataset="commonvoice_ko", VP=1, MP=0, VF=0, MF=-1, FF=0,
        source_model=None, category=None,
        chunk_policy=CHUNK_POLICY_BY_KEY["commonvoice_ko"],
        orig_split=("cv_official" if split else None), split=split,
        group_id=f"commonvoice_ko::{client_id}", source_path=str(f),
    )
    cv_jobs.append(dict(
        in_path=str(f), out_dir=str(AUDIO_OUT / "commonvoice_ko"), base_name=f.stem, label=label,
    ))

cv_rows = run_jobs(cv_jobs, desc="commonvoice_ko")
append_rows_checkpoint(cv_rows)
print("commonvoice_ko 신규 chunk 수:", len(cv_rows), "| 이어서 건너뜀:", cv_skipped)


## 5. MUSAN

`speech/`=실제 음성, `music/`=실제 음악, `noise/`=음성도 음악도 아님(둘 다 판정 대상 아님, `FF=0`).
MUSAN 파일들은 화자/트랙 메타데이터가 파일명에 없는 독립적인 짧은 세그먼트들이라, 화자 단위로 그룹핑할
근거가 없습니다. 따라서 **파일 단위 그룹**(`group_id=파일명`)으로 무작위 분할합니다(7절).

In [ ]:
MUSAN_ROOT = RAW_ROOT / "musan" / "extracted" / "musan"
MUSAN_RULES = {
    "speech": dict(VP=1, MP=0, VF=0, MF=-1, FF=0, chunk_policy=CHUNK_POLICY_BY_KEY["musan/speech"]),
    "music":  dict(VP=0, MP=1, VF=-1, MF=0, FF=0, chunk_policy=CHUNK_POLICY_BY_KEY["musan/music"]),
    "noise":  dict(VP=0, MP=0, VF=-1, MF=-1, FF=0, chunk_policy=CHUNK_POLICY_BY_KEY["musan/noise"]),
}

done = load_done_source_paths()

musan_rows = []
musan_skipped = 0
for category, rule in MUSAN_RULES.items():
    files = sorted((MUSAN_ROOT / category).rglob("*.wav"))[:MAX_FILES_PER_DATASET]
    jobs = []
    for f in files:
        if str(f) in done:
            musan_skipped += 1
            continue
        subsource = f.parent.name  # 예: librivox, us-gov, fma, free-sound ...
        label = dict(
            dataset="musan", **rule,
            source_model=None, category=f"{category}/{subsource}",
            orig_split=None, split=None,
            group_id=f"musan::{f.stem}", source_path=str(f),
        )
        jobs.append(dict(
            in_path=str(f), out_dir=str(AUDIO_OUT / "musan" / category / subsource),
            base_name=f.stem, label=label,
        ))
    new_rows = run_jobs(jobs, desc=f"musan/{category}")
    append_rows_checkpoint(new_rows)  # 카테고리 하나가 끝날 때마다 바로 기록
    musan_rows += new_rows

print("musan 신규 chunk 수:", len(musan_rows), "| 이어서 건너뜀:", musan_skipped)


## 6. FakeMusicCaps

전부 생성모델이 만든 가짜 음악입니다 (`VP=0, MP=1, VF=-1, MF=1, FF=1`). 서브폴더명(`audioldm2`,
`MusicGen_medium`, `musicldm`, `mustango`, `stable_audio_open`)을 `source_model`로 남깁니다.

**중요**: 이 데이터셋은 동일한 원곡(MusicCaps 클립, 파일명이 YouTube video id)을 5개 생성모델이 각각
버전을 만든 구조입니다. 즉 `dHGAXJ9RPJs.wav`가 5개 폴더에 모두 존재합니다. 이걸 그냥 무작위로 파일 단위
분할하면 **같은 원곡의 다른 모델 버전이 train과 test에 동시에** 들어가서 내용 누수(content leakage)가
생깁니다. 그래서 `group_id`를 **원곡 id(파일 stem)** 로 잡아, 같은 원곡의 5개 버전이 항상 같은 split에
속하도록 합니다.

In [ ]:
FMC_ROOT = RAW_ROOT / "fakemusiccaps" / "processed"
done = load_done_source_paths()

fmc_jobs = []
fmc_skipped = 0
for model_dir in sorted(d for d in FMC_ROOT.iterdir() if d.is_dir() and d.name != "__MACOSX"):
    files = sorted(model_dir.glob("*.wav"))[:MAX_FILES_PER_DATASET]
    for f in files:
        if str(f) in done:
            fmc_skipped += 1
            continue
        label = dict(
            dataset="fakemusiccaps", VP=0, MP=1, VF=-1, MF=1, FF=1,
            source_model=model_dir.name, category=None,
            chunk_policy=CHUNK_POLICY_BY_KEY["fakemusiccaps"],
            orig_split=None, split=None,
            group_id=f"fakemusiccaps::{f.stem}", source_path=str(f),
        )
        fmc_jobs.append(dict(
            in_path=str(f), out_dir=str(AUDIO_OUT / "fakemusiccaps" / model_dir.name),
            base_name=f.stem, label=label,
        ))

fmc_rows = run_jobs(fmc_jobs, desc="fakemusiccaps")
append_rows_checkpoint(fmc_rows)
print("fakemusiccaps 신규 chunk 수:", len(fmc_rows), "| 이어서 건너뜀:", fmc_skipped)


## 7. 통합 + train/val/test 최종 배정

`asv_rows`/`cv_rows`/`musan_rows`/`fmc_rows`는 "이번 실행에서 새로 처리한 행"일 뿐이고, 실제 전체 데이터는
2절에서부터 계속 기록해온 체크포인트 파일 `PARTIAL_CSV`에 누적되어 있습니다 (이번 실행분 + 과거에 중단됐던
실행에서 이미 끝낸 분 전부 포함). 그래서 통합은 메모리의 리스트를 합치는 대신 **`PARTIAL_CSV`를 통째로
다시 읽어서** 진행합니다 — 이렇게 해야 여러 번에 걸쳐 나눠 실행했어도 최종 결과가 항상 완전합니다.

지금까지 ASVspoof/Common Voice 공식 split에 속한 행들은 이미 `split`이 채워져 있고, 나머지(Common Voice의
비공식 클립, MUSAN 전체, FakeMusicCaps 전체)는 `split=None`인 상태입니다. 이 None들을 데이터셋별로
`grouped_split`을 적용해 채웁니다.

`grouped_split`은 그룹(화자/원곡/파일)을 무작위로 섞은 뒤, **현재까지 배정된 비율이 목표 비율에 가장
못 미치는 split에** 그룹 전체를 통째로 배정하는 방식입니다(greedy proportional fill). 그룹 크기가 서로
달라도(예: FakeMusicCaps는 그룹당 5개, MUSAN은 그룹당 1개) 최종 비율이 목표(80/10/10)에 정확히 수렴합니다.

In [ ]:
def grouped_split(groups: pd.Series, ratios=SPLIT_RATIOS, seed=SPLIT_SEED):
    """group_id 기준으로 (같은 그룹은 항상 같은 split에 가도록) train/val/test를 배정."""
    sizes = groups.value_counts()
    order = sizes.index.to_numpy().copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(order)

    total = sizes.sum()
    names = ["train", "val", "test"]
    targets = {n: r * total for n, r in zip(names, ratios)}
    assigned = {n: 0 for n in names}

    mapping = {}
    for g in order:
        # 목표 대비 진행률(assigned/target)이 가장 낮은 split에 이 그룹을 통째로 배정
        need = {n: (assigned[n] / targets[n] if targets[n] > 0 else 1.0) for n in names}
        choice = min(names, key=lambda n: need[n])
        mapping[g] = choice
        assigned[choice] += sizes[g]
    return mapping


df = pd.read_csv(PARTIAL_CSV)  # 체크포인트 전체 = 이번 실행 + 과거 중단된 실행에서 처리된 모든 chunk
print("전체 chunk 수 (누적):", df.shape)
print("split 미배정 행 수:", df["split"].isna().sum())

for dataset_name, seed_offset in [("commonvoice_ko", 1), ("musan", 2), ("fakemusiccaps", 3)]:
    mask = (df["dataset"] == dataset_name) & (df["split"].isna())
    if mask.any():
        mapping = grouped_split(df.loc[mask, "group_id"], seed=SPLIT_SEED + seed_offset)
        df.loc[mask, "split"] = df.loc[mask, "group_id"].map(mapping)

assert df["split"].isna().sum() == 0, "split이 배정되지 않은 행이 남아있음"
df = df[COLUMN_ORDER]  # 컬럼 순서 정리 (COLUMN_ORDER는 1절 Config에서 정의)

print(pd.crosstab(df["dataset"], df["split"]))

## 8. 검증(sanity check)

- 라벨 값이 정의된 범위 안에 있는지
- 메타데이터에 적힌 모든 청크 파일이 실제로 디스크에 존재하는지
- 일부 청크를 샘플링해 실제 길이가 정확히 10초(160000 샘플)인지, 샘플레이트가 16kHz인지
- split별 라벨 분포(모든 split에 real/fake, voice/music이 고루 섞여 있는지)

In [ ]:
assert set(df["VP"].unique()) <= {0, 1}
assert set(df["MP"].unique()) <= {0, 1}
assert set(df["VF"].unique()) <= {-1, 0, 1}
assert set(df["MF"].unique()) <= {-1, 0, 1}
assert set(df["FF"].unique()) <= {0, 1}
assert set(df["split"].unique()) <= {"train", "val", "test"}
assert set(df["chunk_policy"].unique()) <= {"single_chunk", "sequential_drop", "noise_conditional"}
assert set(df["is_truncated"].unique()) <= {True, False}

missing = df[~df["filepath"].apply(lambda p: (AUDIO_OUT / p).exists())]
assert missing.empty, f"{len(missing)}개 파일이 메타데이터에는 있지만 디스크에 없음"

# 모든 청크는 정책과 무관하게 항상 정확히 CHUNK_SAMPLES 길이여야 한다
# (single_chunk: crop-or-pad, sequential_drop/noise_conditional: 완전한 청크만 생성).
sample_paths = df["filepath"].sample(min(200, len(df)), random_state=0)
for p in sample_paths:
    y, sr = sf.read(str(AUDIO_OUT / p))
    assert sr == TARGET_SR, (p, sr)
    assert len(y) == CHUNK_SAMPLES, (p, len(y))
print(f"{len(sample_paths)}개 청크 길이/샘플레이트 검증 통과")

print(df.groupby(["dataset", "split", "VP", "MP", "VF", "MF", "FF"], dropna=False).size())


## 9. 메타데이터 저장

혼합 데이터를 쉽게 만들 수 있도록, **오디오 저장 구조(경로)와 split 배정을 분리**했습니다. 즉 실제 wav
파일은 `data/raw`의 원본 하위 구조를 그대로 미러링해서 저장하고 (`audio/<dataset>/...`), train/val/test
정보는 오로지 이 CSV의 `split` 컬럼에만 존재합니다. 이렇게 하면 나중에 split 비율을 바꾸거나 재분할하고
싶을 때 오디오를 다시 처리할 필요 없이 `chunks.csv`만 다시 만들면 됩니다.

또한 VP/MP 기준으로 미리 걸러낸 `voice_pool.csv`(음성 성분이 있는 모든 청크)와 `music_pool.csv`(음악
성분이 있는 모든 청크)를 따로 저장합니다 — 추후 음성+음악 혼합 데이터를 만들 때 이 두 파일에서 같은
`split`끼리 샘플을 뽑아 섞으면 됩니다 (아래 10절 예시 참고).

참고로 `metadata/chunks_partial.csv`(체크포인트 파일)는 그대로 남겨둡니다 — split이 아직 채워지지 않은
중간 상태라 최종 산출물로 쓰면 안 되지만, 지워도 상관없고 감사(audit) 기록 삼아 남겨둬도 무방합니다.

In [ ]:
CONFIG = dict(
    target_sr=TARGET_SR,
    chunk_sec=CHUNK_SEC,
    target_lufs=TARGET_LUFS,
    peak_ceiling_db=PEAK_CEILING_DB,
    split_ratios=dict(zip(["train", "val", "test"], SPLIT_RATIOS)),
    split_seed=SPLIT_SEED,
    max_files_per_dataset=MAX_FILES_PER_DATASET,
    chunk_policy_by_dataset=CHUNK_POLICY_BY_KEY,
    chunk_policy_definitions={
        "single_chunk": (
            "파일당 정확히 1개의 10초 청크만 생성. 10초보다 길면 앞 10초만 사용(뒤는 버림, "
            "is_truncated=True), 10초보다 짧으면 뒤를 0으로 패딩(is_padded=True). 짧은 원본을 "
            "loop해서 반복 패턴을 인위적으로 만들지 않기 위함."
        ),
        "sequential_drop": (
            "10초 단위로 순차 분할하고, 마지막 10초 미만 자투리는 패딩하지 않고 버림 "
            "(청크를 만들지 않음). 원본이 충분히 길어 자투리를 버려도 손실이 미미할 때 사용."
        ),
        "noise_conditional": (
            "원본 길이가 10초 이상이면 sequential_drop과 동일하게, 10초 미만이면 single_chunk와 "
            "동일하게(zero-pad) 처리. 길이 편차가 큰 데이터셋(MUSAN noise)에 사용."
        ),
    },
    n_chunks_total=len(df),
    n_source_files=int(df.groupby("dataset")["source_path"].nunique().sum()),
)

df.to_csv(META_OUT / "chunks.csv", index=False)
for split_name in ["train", "val", "test"]:
    df[df["split"] == split_name].to_csv(META_OUT / f"chunks_{split_name}.csv", index=False)

df[df["VP"] == 1].to_csv(META_OUT / "voice_pool.csv", index=False)
df[df["MP"] == 1].to_csv(META_OUT / "music_pool.csv", index=False)

with open(META_OUT / "preprocessing_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("저장 완료:", META_OUT)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))


## 10. (참고) 추후 음성+음악 혼합 데이터 만들 때 사용 예시

아래 함수는 실제로 오디오를 섞지는 않고, `voice_pool` / `music_pool`에서 **같은 split** 안에서
한 쌍을 뽑는 방법만 보여주는 데모입니다. 실제 믹싱(레벨 매칭, 길이 정렬, 겹치기 등)은 별도 노트북에서
구현하면 되며, 이미 두 풀 모두 16kHz/mono/10초/동일 라우드니스로 표준화되어 있으므로 바로 `y_voice + y_music`
형태로 섞기만 하면 됩니다.

In [ ]:
def sample_mixing_pair(voice_df, music_df, split="train", rng=None):
    """voice_pool/music_pool에서 같은 split의 한 쌍을 무작위로 뽑는 예시 함수."""
    rng = rng or np.random.default_rng()
    v_row = voice_df[voice_df["split"] == split].sample(1, random_state=rng.integers(1 << 30)).iloc[0]
    m_row = music_df[music_df["split"] == split].sample(1, random_state=rng.integers(1 << 30)).iloc[0]
    return v_row, m_row


voice_df = pd.read_csv(META_OUT / "voice_pool.csv")
music_df = pd.read_csv(META_OUT / "music_pool.csv")
v_row, m_row = sample_mixing_pair(voice_df, music_df, split="train")
print("voice:", v_row["filepath"], "| VF(voice fake)=", v_row["VF"])
print("music:", m_row["filepath"], "| MF(music fake)=", m_row["MF"])
print("혼합 시 라벨 규칙: file_fake = 1 if (VF==1 or MF==1) else 0  (음성/음악 중 하나라도 FAKE면 전체 FAKE)")

## 설계 결정과 한계 (Design decisions & limitations)

- **Mono 다운믹스**: 평가 데이터는 mono/stereo가 혼재하지만, 이 파이프라인은 학습 데이터를 모두 mono로
  표준화합니다. 이유: (1) 라우드니스 정규화와 향후 음성+음악 믹싱은 채널 레이아웃이 일치해야 깔끔하게
  계산되고, (2) 실제로 4개 소스 데이터셋(ASVspoof/CommonVoice/MUSAN/FakeMusicCaps) 샘플을 확인한 결과
  전부 이미 mono였습니다. 대신 **추론(inference) 파이프라인에서는 stereo 입력이 들어오면 이 학습 분포와
  맞추기 위해 반드시 mono로 다운믹스**해야 합니다 — 이건 이 노트북이 아니라 추론 코드 쪽에서 처리할 부분입니다.
- **자투리(짧은 마지막 조각) 처리 — v2에서 데이터셋별로 분기**: v1(`data_processed/`, 기존
  `data/metadata/audio_preprocess_16k_mono_10s.ipynb`의 "50% 미만이면 버림" 정책을 이어받아 "무조건
  패딩"으로 단순화한 버전)은 모든 데이터셋에서 자투리를 예외 없이 zero-pad했습니다. 이 방식은 ASVspoof/
  CommonVoice처럼 원본 자체가 짧은 발화인 경우엔 실제 길이 특성을 그대로 보존한다는 장점이 있지만,
  FakeMusicCaps(생성 자체가 약 10.0~10.24초)에서는 거의 무음(9.7~9.8초 패딩)인 "가짜 두 번째 청크"를
  만들어냈고, MUSAN처럼 원본이 충분히 긴 데이터셋에서는 자투리를 굳이 패딩할 필요가 없었습니다.
  v2는 이를 데이터셋 특성별 3가지 정책(`single_chunk`/`sequential_drop`/`noise_conditional`, 1절 표 참고)
  으로 분기해 개선했습니다 — **어느 경우에도 원본을 loop(반복)해서 패딩하지는 않습니다.** 짧은 원본은
  zero-pad로 실제 길이를 보존하고, 충분히 긴 원본은 자투리를 버려 무음이 섞인 청크를 만들지 않습니다.
  v1 결과물(`data_processed/`)은 그대로 남겨두고, v2는 별도 폴더(`data_processed_v2/`)에 생성합니다.
- **라우드니스 목표치(-23 LUFS)**: EBU R128 방송 표준값을 기본으로 사용했습니다. 다른 값이 낫다고 판단되면
  `TARGET_LUFS` 하나만 바꾸면 전체 파이프라인에 반영됩니다.
- **전화채널(telephone-channel) 오디오**: 평가 데이터 특성상 일부 샘플이 전화채널(좁은 대역폭, 코덱 아티팩트)
  오디오라고 명시되어 있는데, 이번 4개 학습 데이터셋에는 그런 성격의 데이터가 없습니다. 이 노트북에서는
  이를 인위적으로 만들어내지 않았습니다 — 이건 "학습 데이터 전처리"보다는 "학습 중 augmentation"(예: 대역제한
  필터 + 코덱 시뮬레이션을 랜덤 적용)의 영역이라고 판단했기 때문입니다. 향후 augmentation 단계에서 반영하는 걸
  권장합니다.
- **split 재현성**: `SPLIT_SEED`와 각 데이터셋의 그룹핑 기준(화자/원곡/파일)을 고정해뒀기 때문에, 같은 설정으로
  다시 실행하면 항상 동일한 split이 나옵니다. `preprocessing_config.json`에 실행 시점의 설정을 기록해둡니다.
- **실행 시간**: 참고로 과거 동일 규모(ASVspoof 약 12만 파일 포함) 전체를 단일 프로세스로 처리했을 때 총
  3시간 가까이 걸렸습니다. 이 노트북은 `ProcessPoolExecutor`로 병렬화했지만 이 서버는 4코어뿐이라 극적으로
  빨라지진 않으니, 먼저 `MAX_FILES_PER_DATASET`를 작은 값으로 설정해 전체 파이프라인이 정상 동작하는지
  확인한 뒤 `None`으로 되돌려 전체 실행하는 걸 권장합니다. 대량 실행은 이 노트북 대신 동일 로직의
  `preprocess_data_v2.py`를 tmux 세션에서 백그라운드로 돌리는 걸 권장합니다 (연결이 끊겨도 계속 실행됨).
- **체크포인트/재개(resume)**: 이 서버는 공유 서버라 접속이 끊기거나 커널이 죽을 수 있습니다. 그래서
  파일 하나가 처리될 때마다(데이터셋 전체가 끝날 때가 아니라) 바로 `chunks_partial.csv`에 결과를
  append하고, 각 데이터셋 섹션은 job을 만들기 전에 이미 끝난 `source_path`를 걸러냅니다. 즉 **중간에
  끊겨도 노트북을 처음부터 다시 실행하면 되고, 이미 처리된 파일은 자동으로 건너뛰어 몇 초~몇 분 안에
  중단 지점까지 따라잡습니다.** 단, 한 파일을 처리하는 도중(그 파일의 마지막 chunk가 저장되기 전) 정확히
  그 순간에 죽으면 그 파일 하나만 재처리되며, 이미 써진 chunk 파일들은 재실행 시 동일한 내용으로 덮어써져
  손상 없이 안전합니다.
